## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value! This is the start of a lab that will last 2 days.

And we're going to hand-build an Agent Loop without any Agent Framework..

### First, some prep

In the folder `twin` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours! You should be able to download it from your LinkedIn profile; go to your profile page use the menu under your name. If you don't have access to this feature, any PDF such as your resume is great.

I've also made a file called `summary.txt` in `twin` - please read it and update it to reflect you.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. If you're wondering how you would select packages for your own projects, please see Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> page.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
import ollama #changed to ollama to accommadate my laptop
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [8]:
load_dotenv(override=True)
client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [2]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [3]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

## Sidebar: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...

__For more, see my companion course AI Engineer Core Track (first week)__

In [6]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [10]:
response = openai.chat.completions.create(model="llama3.2", messages=messages)
print(response.choices[0].message.content)

Hi Ed! It's nice to meet you. Is there something I can help you with or would you like to chat for a bit?


In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [7]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"},
    {"role": "assistant", "content": "Well hi there, Ed. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [12]:
response = openai.chat.completions.create(model="llama3.2", messages=messages)
print(response.choices[0].message.content)

Come on, Ed! Don't be awkward here. Your name is still Ed. (If I'd forgotten it and wasn't so invested in being snarky, that is)


## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [4]:
system_prompt = f"""

# Your role

You are an AI digital twin assistant running on a website.
You represent the person whose website you are on, but you are NOT the person.
Your role is to provide information about this person's career, background, skills, and experience.

When the user first starts the conversation, briefly introduce yourself as the AI digital twin and explain what you can help with.
Do not provide a detailed biography, resume summary, or list of experiences unless the user asks for more information.
The first response should:
1. State that you are an AI digital twin of Wenwan Xu.
2. Briefly explain that you can answer questions about Wenwan Xu's career, education, skills, projects, and professional experiences.
3. Invite the user to ask a question.

Keep the first response under 50 words.


Always refer to the person you represent using third-person pronouns ("she/her") or their name.
Never speak as if you are the person. Do not use first-person pronouns ("I", "my", "me") when describing the person's experiences, achievements, education, or background.

For example:
- Correct: "XX (Name of the person) has experience ..."
- Incorrect: "I have experience ..."

You may use "I" only when referring to yourself as the AI assistant.
For example:
- "I am XX's AI digital twin, and I can help answer questions about her professional background."

# Context

Here are the details of the person you are representing:

{summary}

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Identity (do not change)

The person you represent is:
Name: Wenwan Xu

Important:
- Always use the exact name "Wenwan Xu" or "Wenwan" when referring to this person.
- Never change, shorten, or replace the name with another name.
- If referring with pronouns, use she/her.
- Do not invent other identities or names.

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.
Keep every response under 100 words unless the user explicitly asks for more detail.
The person being represented is described in third person.

Always stay in character as the digital twin of the person you are representing. Represent the person while clearly state yourself as an AI agent.

IMPORTANT: Before answering, verify that names, organizations, education, and experiences match the provided context exactly.
Do not infer or modify personal information. If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know. 
"""

In [ ]:
display(Markdown(system_prompt))

In [7]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "/no_think\nHi - please tell me about yourself"},
]

In [ ]:
response = client.chat.completions.create(model="qwen3.5:4b", messages=messages)
display(Markdown(response.choices[0].message.content))

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="qwen3.5:9b", messages=messages)
    return response.choices[0].message.content

In [44]:
def chat_fn(message, history):
    history = history[-6:]
    messages = [
        {
            "role": "system",
            "content": system_prompt
        }
    ]

    for h in history:
        role = h["role"]
        content = h["content"]

        # Convert Gradio message blocks to plain text
        if isinstance(content, list):
            content = "".join(
                item["text"]
                for item in content
                if item.get("type") == "text"
            )

        messages.append({
            "role": role,
            "content": content
        })

    messages.append({
        "role": "user",
        "content": message
    })

    response = chat(
        model="qwen3.5:4b",
        think=False,
        keep_alive="30m",
        options={
        "temperature": 0.4
    },
        messages=messages,
    )

    return response.message.content

In [ ]:
chat_fn("Please summarize who you are", [])

## NOTE for those not using OpenAI models

If you're using models other than OpenAI, then you might need to insert this line at the top of chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [ ]:
gr.ChatInterface(chat_fn).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, req

# And now - TOOLS!

Let's start with a function...

In [5]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [6]:
record_email_tool("test@testy.com")

Tool called to record an email: test@testy.com


'Email received'

## Step 1 - write some json to describe the tool


In [7]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [22]:
def record_email_tool(email):
    try:
        with open("emails.txt", "a") as f:
            f.write(email + "\n")

        return {
            "success": True,
            "message": f"Email {email} has been recorded."
        }

    except Exception as e:
        return {
            "success": False,
            "message": str(e)
        }

In [8]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [50]:
tools

[{'type': 'function',
  'function': {'name': 'record_email_tool',
   'description': 'Use this tool to record that a user provided their email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'}},
    'required': ['email'],
    'additionalProperties': False}}}]

## Step 2 - a new chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE: If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [14]:
import ollama
import json

def chat_fn(message, history):

    history = history[-6:]

    messages = [
        {
            "role": "system",
            "content": system_prompt
        }
    ]

    for h in history:
        role = h["role"]
        content = h["content"]

        if isinstance(content, list):
            content = "".join(
                item["text"]
                for item in content
                if item.get("type") == "text"
            )

        messages.append({
            "role": role,
            "content": content
        })

    messages.append({
        "role": "user",
        "content": message
    })


    response = ollama.chat(
        model="qwen3.5:4b",
        messages=messages,
        tools=tools,
        think=False,
        keep_alive="30m",
        options={
            "temperature": 0.4
        }
    )


    if response.message.tool_calls:

        tool_call = response.message.tool_calls[0]

        args = tool_call.function.arguments
        email = args.get("email")

        record_email_tool(email)


        messages.append(response.message)

        messages.append({
            "role": "tool",
            "content": "Email recorded"
        })


        response = ollama.chat(
            model="qwen3.5:4b",
            messages=messages,
            tools=tools,
            think=False
        )


    return response.message.content

In [15]:
gr.ChatInterface(chat_fn).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called to record an email: a@test.com


c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [25]:
def chat_fn(message, history):

    history = history[-6:]

    messages = [
        {
            "role": "system",
            "content": system_prompt
        }
    ]

    for h in history:
        role = h["role"]
        content = h["content"]

        if isinstance(content, list):
            content = "".join(
                item["text"]
                for item in content
                if item.get("type") == "text"
            )

        messages.append({
            "role": role,
            "content": content
        })

    messages.append({
        "role": "user",
        "content": message
    })


    response = ollama.chat(
        model="qwen3.5:4b",
        messages=messages,
        tools=tools,
        think=False,
        keep_alive="30m",
        options={
            "temperature": 0.4
        }
    )


    while response.message.tool_calls:

        # Add assistant tool-call message
        messages.append(response.message)

        for tool_call in response.message.tool_calls:

            args = tool_call.function.arguments

            if isinstance(args, str):
                args = json.loads(args)

            email = args.get("email")

            result = record_email_tool(email)

            print("Tool called:", email)
            print("Tool result:", result)

            messages.append({
                "role": "tool",
                "content": json.dumps(result)
            })


        response = ollama.chat(
            model="qwen3.5:4b",
            messages=messages,
            tools=tools,
            think=False,
            keep_alive="30m",
            options={
                "temperature": 0.4
            }
        )


    return response.message.content

In [26]:
gr.ChatInterface(chat_fn).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called: b@test.com
Tool result: {'success': True, 'message': 'Email b@test.com has been recorded.'}
Tool called: c@test.com
Tool result: {'success': True, 'message': 'Email c@test.com has been recorded.'}
Tool called: d@test.com
Tool result: {'success': True, 'message': 'Email d@test.com has been recorded.'}


c:\Users\xec9cp\Git\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.
            </span>
        </td>
    </tr>
</table>